In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Atenção texto × séries temporais: sinal ou apenas cobertura?

**Pergunta:** manchetes verificadas acrescentam sinal à previsão numérica de retornos relativos? Esta é uma demonstração de pesquisa, **não uma estratégia com alfa comprovado**. O notebook apresenta a atenção residual já implementada e executada, usa embeddings MOMENT-1-small e MiniLM gerados na execução anterior e lê resultados existentes para abrir rapidamente no Colab. Se os resultados ainda não existirem, treina os dois resíduos reais em uma GPU; nunca inventa métricas.

**Roteiro:** proveniência → disponibilidade das notícias → arquitetura → ablações → validação → previsão e carteira → diagnóstico da contribuição do texto → limitações.

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
from IPython.display import Image, display
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
SOURCE = DATA / 'fast_fusion_demo_results'
for filename in ('fast_demo_panel.npz', 'fast_demo_events.npz', 'fast_demo_provenance.json', 'ts_only.pt', 'demo_titles.jsonl.gz'):
    if not (SOURCE / filename).is_file():
        raise FileNotFoundError(f'Execute antes fast_fusion_demo_colab.ipynb: {SOURCE / filename}')
{'fonte': str(SOURCE), 'repositório': str(REPO)}

## 1. Dados, período e rastreabilidade

O painel guarda **retornos observados e embeddings numéricos gerados naquela execução**; o arquivo de eventos guarda embeddings de manchetes e atributos léxicos. As manchetes têm a data de decisão atrasada em um pregão quando o horário de publicação não é seguro. A validação abaixo confere que o painel, os eventos e o checkpoint numérico pertencem à mesma execução. Nenhum encoder, LLM ou notícia adicional é baixado.

In [ ]:
from src.fusion.fast_residual_experiment import load_saved_demo, run_experiment
from src.fusion.text_attention import TextEventDataset
panel, events, baseline_checkpoint, provenance = load_saved_demo(SOURCE)
display(pd.DataFrame([{'fonte numérica': provenance['moment_model_id'], 'fonte textual': provenance['text_model_id'], 'embeddings numéricos': provenance['moment_embeddings'], 'manchetes': provenance['article_embeddings'], 'ativos': len(panel['tickers']), 'geração UTC': provenance['created_at_utc']}]))
assert provenance['fresh_inference'] and not provenance['old_embedding_cache_used']

## 2. Alvo e informação disponível

Para cada ativo e fechamento de decisão $t$, o alvo é o retorno dos próximos **cinco pregões** menos a média dos retornos futuros dos mesmos oito ativos. Esse retorno futuro é **somente rótulo/avaliação**, nunca entrada do modelo. As entradas são os últimos 64 retornos já conhecidos e manchetes disponíveis até $t$, no máximo oito das últimas cinco sessões. Treino, validação e teste são cronológicos, com purga dos rótulos de cinco dias nas fronteiras. O teste usa todos os pregões, inclusive os sem manchete; ‘com notícia’ não significa notícia publicada no próprio dia.

In [ ]:
coverage = []
for split in ('train', 'val', 'test'):
    dataset = TextEventDataset(panel, events, split)
    dates = [panel['arrays'][int(ti)]['dates'][int(j)] for ti, j in dataset.samples]
    with_news = sum(bool(dataset._news(int(ti), int(j))[1].any()) for ti, j in dataset.samples)
    coverage.append({'partição': split, 'primeira decisão': str(pd.Timestamp(min(dates)).date()), 'última decisão': str(pd.Timestamp(max(dates)).date()), 'amostras': len(dataset), 'com notícia nos últimos 5 pregões': with_news, 'cobertura (%)': 100 * with_news / len(dataset)})
coverage = pd.DataFrame(coverage).set_index('partição')
display(coverage.round(2))

## 3. Estratégia: atenção cruzada residual

O **modelo numérico pré-treinado fica congelado**. Sua representação das séries temporais é a consulta de uma `MultiheadAttention` de quatro cabeças; chaves/valores são até oito manchetes elegíveis e uma chave nula válida. O ramo textual usa o MiniLM mais atributos léxicos, novidade e idade. Uma porta sigmoidal modula a correção limitada: $\hat y_t = \hat y_t^{\mathrm{num}} + I_{\mathrm{notícia}} \; 0{,}02 \;\sigma(g_t)\tanh(r_t)$. A última camada do residual começa em zero: antes do treino a previsão é exatamente a numérica. Sem notícia recente, o residual é zero.

**Três controles no mesmo calendário:** numérico congelado; numérico + atenção apenas a idade/contagem das manchetes; numérico + atenção a conteúdo, atributos e idade. A comparação texto × idade/contagem é a ablação necessária para investigar **informação semântica**, em vez de confundir presença de notícia com significado. A previsão de excesso zero é referência de erro, não carteira.

## 4. Treino ou reutilização auditada

Se houver resultados completos com **a mesma proveniência** e custo de 10 bps, o notebook os lê imediatamente (inclusive em Colab CPU). Caso contrário, treina os dois resíduos na GPU, em saída separada, sem recalcular embeddings ou descongelar o modelo numérico. Somente o período de validação escolhe a época; o teste 2022–23 não participa da escolha.

In [ ]:
required = ('run_config.json', 'signal_metrics.csv', 'portfolio_metrics.csv', 'equity_curves.csv', 'metadata_history.csv', 'text_residual_history.csv', 'predictions_ts_only.csv', 'predictions_metadata.csv', 'predictions_text_residual.csv', 'residual_vs_bh.png')
candidates = (DATA / 'fast_residual_attention_results', DATA / 'attention_presentation_results')
ready = [folder for folder in candidates if all((folder / name).is_file() for name in required)]
if ready:
    OUT = ready[0]
    mode = 'Resultados reais da execução anterior'
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Sem resultados completos: ative T4/L4 no Colab para treinar o residual')
    OUT = candidates[1]
    run_experiment(SOURCE, OUT, cost_bps=10)
    mode = 'Treino real concluído nesta execução'
config = json.loads((OUT / 'run_config.json').read_text(encoding='utf-8'))
if config['embedding_provenance'] != provenance or config['cost_bps'] != 10 or config['models'] != ['ts_only', 'metadata', 'text_residual', 'zero', 'buy_hold']:
    raise ValueError('Resultados incompatíveis com os embeddings e controles carregados')
{'modo': mode, 'diretório': str(OUT), 'épocas máximas': config['max_epochs'], 'residual máximo': config['max_residual']}

## 5. Diagnóstico de validação

O erro de validação considera **somente linhas com manchete**, pois dias sem notícia devolvem o modelo numérico congelado. O gráfico permite verificar se continuar treinando melhora o treino mas piora a validação; não compare seu valor diretamente ao histórico numérico calculado em outra população de linhas.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.backtest.visualization import BG, PANEL, CYAN, PURPLE, WHITE
histories = {name: pd.read_csv(OUT / f'{name}_history.csv') for name in ('metadata', 'text_residual')}
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
for name, label, color in (('metadata', 'Idade/contagem', PURPLE), ('text_residual', 'Manchetes', CYAN)):
    history = histories[name]
    ax.plot(history['epoch'], history['val_mse'], marker='o', linewidth=2.2, label=label, color=color)
ax.set_title('Erro de validação: resíduos com notícias', fontsize=16, color='white')
ax.set_xlabel('Época', fontsize=14, color='white')
ax.set_xticks(sorted(set(histories['metadata']['epoch']) | set(histories['text_residual']['epoch'])))
ax.set_ylabel('MSE na validação', fontsize=14, color='white')
ax.tick_params(colors='white', labelsize=12)
ax.grid(alpha=0.12)
ax.legend(facecolor=PANEL, labelcolor='white', edgecolor=WHITE, fontsize=14)
fig.tight_layout()
fig.savefig(OUT / 'presentation_validation.png', dpi=300, facecolor=BG, edgecolor='none')
display(fig)
plt.close(fig)
{'melhor época (metadados)': int(histories['metadata'].loc[histories['metadata']['val_mse'].idxmin(), 'epoch']), 'melhor época (texto)': int(histories['text_residual'].loc[histories['text_residual']['val_mse'].idxmin(), 'epoch'])}

## 6. Evidência fora da amostra: previsão e carteira

MSE, MAE e IC Spearman avaliam o retorno relativo futuro de cinco pregões. A carteira transforma as previsões em pesos **long-only** e rebalanceia a cada cinco pregões após o fechamento. Retornos são líquidos de 10 bps por valor negociado, inclusive entrada; a referência compra os mesmos oito ativos com pesos iguais e mantém. Compare o texto também à idade/contagem: superar apenas uma fusão anterior que reaprendeu a cabeça numérica **não prova** contribuição da linguagem.

In [ ]:
signal = pd.read_csv(OUT / 'signal_metrics.csv', index_col=0)
portfolio = pd.read_csv(OUT / 'portfolio_metrics.csv', index_col=0)
equity = pd.read_csv(OUT / 'equity_curves.csv', parse_dates=['date']).set_index('date')
assert len(equity) > 0 and all(np.isclose(equity[name].iloc[-1], 1 + portfolio.loc[name, 'return_pct'] / 100) for name in portfolio.index)
display(signal[['mse_5d', 'mae_5d', 'ic_spearman', 'n_amostras', 'com_evento']].round(8))
display(portfolio[['return_pct', 'bh_return_pct', 'excess_pct', 'alpha_annual_pct', 'beta', 'sharpe', 'turnover_total']].round(6))
display(Image(filename=str(OUT / 'residual_vs_bh.png')))

## 7. Quanto a atenção realmente mudou?

Uma curva visualmente próxima à referência pode esconder um efeito ínfimo. Abaixo, as previsões são alinhadas por **ticker e data**; a correção média absoluta e a quantidade de pregões cujo ativo de maior pontuação mudou medem a materialidade do residual, sem reinterpretar pesos de atenção como causalidade. Tolerância de $10^{-6}$ em dias sem notícia permite diferenças numéricas de GPU entre tamanhos de lote.

In [ ]:
forecasts = {name: pd.read_csv(OUT / f'predictions_{name}.csv', parse_dates=['date']) for name in ('ts_only', 'metadata', 'text_residual')}
reference = forecasts['ts_only']
if (SOURCE / 'predictions_ts_only.csv').is_file():
    original = pd.read_csv(SOURCE / 'predictions_ts_only.csv', parse_dates=['date'])
    if not original[['ticker', 'date']].equals(reference[['ticker', 'date']]) or not np.allclose(original['score'], reference['score'], atol=1e-6):
        raise ValueError('O checkpoint numérico não reproduz as previsões da execução original')
diagnostics = []
leaders = reference.loc[reference.groupby('date')['score'].idxmax()].set_index('date')['ticker']
for name in ('metadata', 'text_residual'):
    frame = forecasts[name]
    if not frame[['ticker', 'date']].equals(reference[['ticker', 'date']]) or not frame['has_event'].equals(reference['has_event']):
        raise ValueError('A ablação mudou as datas ou a cobertura do teste')
    delta = frame['score'] - reference['score']
    if delta.loc[~reference['has_event']].abs().max() > 1e-6:
        raise ValueError('O residual modificou uma previsão sem notícia')
    current_leaders = frame.loc[frame.groupby('date')['score'].idxmax()].set_index('date')['ticker']
    diagnostics.append({'modelo': name, 'correção média absoluta': delta.abs().mean(), 'correção máxima absoluta': delta.abs().max(), 'pregões com líder diferente': int((leaders != current_leaders).sum())})
display(pd.DataFrame(diagnostics).set_index('modelo').round(8))
{'diferença texto − metadados (pontos percentuais)': round(float(portfolio.loc['text_residual', 'return_pct'] - portfolio.loc['metadata', 'return_pct']), 6), 'MSE da previsão zero': float(signal.loc['zero', 'mse_5d'])}

## 8. O que pode ser apresentado — e o que não pode

**Implementação demonstrada:** atenção cruzada genuína, modalidade numérica congelada, correção textual condicional, ablação de metadados, mesma amostra e mesmo comprar e manter. **Resultado não demonstrado:** alfa atribuível ao significado das notícias. O universo foi escolhido após observar a cobertura e o teste de 2022–23 já tinha sido inspecionado. O alvo de cinco pregões gera observações sobrepostas; manchete ausente não implica ausência de notícias; a fonte conhecida falha após novembro de 2023.

**Resposta à banca:** se o texto não superar consistentemente idade/contagem, o estudo mostra que a arquitetura funciona e que a fonte/rotulação ainda não fornece sinal semântico comprovado — não que atenção, por si só, gera retorno. Próximo experimento: obter cobertura auditada de um período **novo** e repetir as três comparações sem ajustar hiperparâmetros no teste já visto.

In [ ]:
{'execução': mode, 'período do teste': (str(equity.index.min().date()), str(equity.index.max().date())), 'texto acima de metadados no retorno líquido': bool(portfolio.loc['text_residual', 'return_pct'] > portfolio.loc['metadata', 'return_pct']), 'texto acima de metadados no MSE': bool(signal.loc['text_residual', 'mse_5d'] < signal.loc['metadata', 'mse_5d']), 'retorno do texto (%)': round(float(portfolio.loc['text_residual', 'return_pct']), 4), 'retorno comprar e manter (%)': round(float(portfolio.loc['buy_hold', 'return_pct']), 4), 'relatório': str(OUT / 'results.md')}